# RiskLens — Data Exploration (P02)
**CS 4300 Spring 2026**

This notebook covers:
1. Downloading the FNSPID dataset
2. Loading and describing the sample
3. Preprocessing (tokenization, stopword removal)
4. IR Statistics: TF histogram, IDF distribution, document length distribution
5. Top terms sanity check
6. Concrete data examples


## 0. Install & Import Dependencies

In [7]:
# Run this cell once to install dependencies
# !pip install huggingface_hub pandas matplotlib seaborn nltk scikit-learn

In [3]:
import os
import json
import math
import re
import zipfile
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

STOP_WORDS = set(stopwords.words('english'))

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

print('All imports successful.')

All imports successful.


## 1. Load & Take a Subset of Dataset

In [7]:
# Load the full CSV — use chunksize if memory is a concern on your machine
# For the PoC/P02 we load a 50k-row sample to keep things fast and add it to a CSV for submission
from pathlib import Path

SAMPLE_SIZE = 50000
base_dirs = [Path.cwd(), Path.cwd().parent]
candidate_files = [
    'data/nasdaq_external_data.csv',
    'data/nasdaq_exteral_data.csv',  # filename from download script
    'data/fnspid/Stock_news/nasdaq_exteral_data.csv'
]

CSV_PATH = None
for base in base_dirs:
    for rel_path in candidate_files:
        candidate = base / rel_path
        if candidate.exists():
            CSV_PATH = candidate
            break
    if CSV_PATH is not None:
        break

if CSV_PATH is None:
    raise FileNotFoundError(f'Could not find dataset. Checked: {candidate_files}')

out_dir = CSV_PATH.parent
SAMPLE_PATH = out_dir / 'fnspid_sample_gradescope.csv'
ZIP_PATH = out_dir / 'fnspid_sample.zip'

df_full = pd.read_csv(CSV_PATH, nrows=SAMPLE_SIZE)
print(f'Using CSV path: {CSV_PATH.resolve()}')
print(f'Loaded {len(df_full):,} rows')
print(f'Columns: {list(df_full.columns)}')

# ── Create a representative sample (random so it's not all the same ticker) ──
sample_df = df_full.sample(n=min(5000, len(df_full)), random_state=42)

# ── Save to CSV ───────────────────────────────────────────────────────────────
out_dir.mkdir(parents=True, exist_ok=True)
sample_df.to_csv(SAMPLE_PATH, index=False)

size_mb = os.path.getsize(SAMPLE_PATH) / (1024 * 1024)
print(f'Sample CSV: {size_mb:.2f} MB ({len(sample_df):,} rows)')

# ── If still under 10MB, expand until we get close to the limit ──────────────
n = 5000
while size_mb < 9.0 and n < len(df_full):
    n += 1000
    sample_df = df_full.sample(n=n, random_state=42)
    sample_df.to_csv(SAMPLE_PATH, index=False)
    size_mb = os.path.getsize(SAMPLE_PATH) / (1024 * 1024)

print(f'Final sample: {size_mb:.2f} MB ({n:,} rows)')

# ── Zip it ────────────────────────────────────────────────────────────────────
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(SAMPLE_PATH, arcname='fnspid_sample_gradescope.csv')

zip_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f'Zip created: {zip_mb:.2f} MB → {ZIP_PATH}')
print('Ready to submit to Gradescope.')


FileNotFoundError: [Errno 2] No such file or directory: './data/nasdaq_external_data.csv'

In [ ]:
# Inspect the first few rows
df_full.head(3)

In [ ]:
# ── Identify the text and ticker columns ──────────────────────────────────────
# FNSPID columns are typically: Date, Stock Symbol, Title, Article, Lable (sentiment)
# Rename for clarity if needed
col_map = {}
for col in df_full.columns:
    lower = col.lower().strip()
    if 'title' in lower:       col_map[col] = 'title'
    elif 'article' in lower:   col_map[col] = 'article'
    elif 'symbol' in lower or 'stock' in lower or 'ticker' in lower: col_map[col] = 'ticker'
    elif 'date' in lower:      col_map[col] = 'date'
    elif 'label' in lower or 'lable' in lower: col_map[col] = 'sentiment'

df = df_full.rename(columns=col_map)
print('Renamed columns:', list(df.columns))

In [ ]:
# ── Basic corpus statistics ───────────────────────────────────────────────────
# Combine title + article into a single text field
title_col   = 'title'   if 'title'   in df.columns else df.columns[2]
article_col = 'article' if 'article' in df.columns else df.columns[3]
ticker_col  = 'ticker'  if 'ticker'  in df.columns else df.columns[1]
date_col    = 'date'    if 'date'    in df.columns else df.columns[0]

df['text'] = df[title_col].fillna('') + ' ' + df[article_col].fillna('')
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

total_docs       = len(df)
unique_tickers   = df[ticker_col].nunique()
date_min         = df[date_col].min()
date_max         = df[date_col].max()
null_text        = df['text'].str.strip().eq('').sum()
avg_article_len  = df[article_col].fillna('').apply(lambda x: len(x.split())).mean()

print('=' * 50)
print(f'Total documents (sample):  {total_docs:,}')
print(f'Unique tickers:            {unique_tickers:,}')
print(f'Date range:                {date_min.date()} → {date_max.date()}')
print(f'Missing/empty text:        {null_text:,}')
print(f'Avg article length (words):{avg_article_len:.1f}')
print('=' * 50)

In [ ]:
# ── Concrete examples: show 5 real rows ──────────────────────────────────────
print('=== CONCRETE DATA EXAMPLES ===')
sample_rows = df[[date_col, ticker_col, title_col, article_col]].dropna().sample(5, random_state=42)
for _, row in sample_rows.iterrows():
    print(f"\nDate:    {row[date_col]}")
    print(f"Ticker:  {row[ticker_col]}")
    print(f"Title:   {row[title_col]}")
    print(f"Snippet: {str(row[article_col])[:200]}...")
    print('-' * 60)

## 3. Preprocessing
Tokenize, lowercase, remove stopwords and punctuation.

In [ ]:
def preprocess(text: str) -> list[str]:
    """Lowercase, remove non-alpha chars, tokenize, remove stopwords."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return tokens

print('Preprocessing corpus...')
df['tokens'] = df['text'].apply(preprocess)
df['doc_length'] = df['tokens'].apply(len)
print(f'Done. Example tokens: {df["tokens"].iloc[0][:15]}')

## 4. Build the Inverted Index & Compute TF/IDF

In [ ]:
# ── Term Frequency (TF) across the entire corpus ──────────────────────────────
corpus_tf = Counter()
for tokens in df['tokens']:
    corpus_tf.update(tokens)

print(f'Unique terms in corpus: {len(corpus_tf):,}')
print(f'Total term occurrences: {sum(corpus_tf.values()):,}')
print(f'Top 10 terms: {corpus_tf.most_common(10)}')

In [ ]:
# ── Inverted Index & Document Frequency (DF) ──────────────────────────────────
inverted_index = defaultdict(set)   # term -> set of doc indices
doc_tf_index   = []                  # per-doc term freq dict

for idx, tokens in enumerate(df['tokens']):
    tf = Counter(tokens)
    doc_tf_index.append(tf)
    for term in tf:
        inverted_index[term].add(idx)

N = len(df)
# IDF = log(N / df(t))
idf = {term: math.log(N / len(doc_ids)) 
       for term, doc_ids in inverted_index.items()}

print(f'Inverted index built: {len(inverted_index):,} unique terms')
print(f'Sample IDF values:')
for term in ['stock', 'earnings', 'semiconductor', 'ransomware', 'inflation']:
    if term in idf:
        print(f'  {term}: IDF = {idf[term]:.3f}')

## 5. Plots

### Plot 1 — Document Length Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

doc_lengths = df['doc_length']
# Cap at 99th percentile so extreme outliers don't squash the plot
cap = int(doc_lengths.quantile(0.99))
clipped = doc_lengths.clip(upper=cap)

ax.hist(clipped, bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(doc_lengths.median(), color='tomato', linestyle='--',
           linewidth=1.5, label=f'Median = {doc_lengths.median():.0f} tokens')
ax.axvline(doc_lengths.mean(), color='orange', linestyle='--',
           linewidth=1.5, label=f'Mean = {doc_lengths.mean():.0f} tokens')

ax.set_title('Document Length Distribution (tokens after preprocessing)', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Tokens per Document')
ax.set_ylabel('Number of Documents')
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('./data/plot_doc_length_dist.png', bbox_inches='tight')
plt.show()
print('Saved: plot_doc_length_dist.png')

### Plot 2 — Term Frequency (TF) Histogram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

tf_values = list(corpus_tf.values())

# Left: raw TF (log scale to show the long tail)
axes[0].hist(tf_values, bins=80, color='steelblue', edgecolor='white', linewidth=0.3, log=True)
axes[0].set_title('Term Frequency Distribution (log scale)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Term Frequency (count across corpus)')
axes[0].set_ylabel('Number of Terms (log scale)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Right: zoomed in on low-frequency terms (the long tail bulk)
low_freq = [v for v in tf_values if v <= 50]
axes[1].hist(low_freq, bins=50, color='mediumseagreen', edgecolor='white', linewidth=0.3)
axes[1].set_title('Term Frequency — Low Frequency Terms (TF ≤ 50)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Term Frequency')
axes[1].set_ylabel('Number of Terms')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

pct_hapax = sum(1 for v in tf_values if v == 1) / len(tf_values) * 100
fig.suptitle(f'Zipfian Long-Tail Distribution  |  {pct_hapax:.1f}% of terms appear only once (hapax legomena)',
             fontsize=11, y=1.02)

plt.tight_layout()
plt.savefig('./data/plot_tf_histogram.png', bbox_inches='tight')
plt.show()
print('Saved: plot_tf_histogram.png')

### Plot 3 — IDF Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

idf_values = list(idf.values())
ax.hist(idf_values, bins=70, color='mediumpurple', edgecolor='white', linewidth=0.3)

# Annotate the two extremes
ax.axvline(x=0.5, color='tomato', linestyle='--', linewidth=1.2,
           label='Low IDF → common terms (low discriminability)')
ax.axvline(x=math.log(N) * 0.85, color='darkorange', linestyle='--', linewidth=1.2,
           label='High IDF → rare terms (high discriminability)')

ax.set_title('IDF Distribution Across Vocabulary', fontsize=13, fontweight='bold')
ax.set_xlabel('IDF Score  [ log(N / df(t)) ]')
ax.set_ylabel('Number of Terms')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('./data/plot_idf_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: plot_idf_distribution.png')

### Plot 4 — Top 25 Most Frequent Terms (after stopword removal)

In [ ]:
# Extra finance-specific stopwords that aren't in NLTK but aren't meaningful for IR
FINANCE_STOPWORDS = {'said', 'also', 'would', 'could', 'one', 'new', 'year',
                     'inc', 'corp', 'ltd', 'llc', 'co', 'reuters', 'bloomberg'}

filtered_tf = {term: count for term, count in corpus_tf.items()
               if term not in FINANCE_STOPWORDS and len(term) > 2}

top_terms = sorted(filtered_tf.items(), key=lambda x: x[1], reverse=True)[:25]
terms, counts = zip(*top_terms)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(terms[::-1], counts[::-1], color='steelblue', edgecolor='white')

# Color finance-domain terms differently
finance_domain = {'stock', 'shares', 'earnings', 'revenue', 'market', 'quarter',
                  'company', 'billion', 'million', 'growth', 'price', 'profit',
                  'sales', 'financial', 'investment', 'chip', 'semiconductor'}
for bar, term in zip(bars, terms[::-1]):
    if term in finance_domain:
        bar.set_color('mediumseagreen')

ax.set_title('Top 25 Most Frequent Terms After Stop Word Removal\n'
             '(green = finance-domain terms)', fontsize=12, fontweight='bold')
ax.set_xlabel('Corpus Frequency')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('./data/plot_top_terms.png', bbox_inches='tight')
plt.show()
print('Saved: plot_top_terms.png')

### Plot 5 — Top 20 Most Mentioned Tickers

In [ ]:
ticker_counts = df[ticker_col].value_counts().head(20)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(ticker_counts.index, ticker_counts.values, color='steelblue', edgecolor='white')
ax.set_title('Top 20 Most Mentioned Tickers in FNSPID Sample', fontsize=13, fontweight='bold')
ax.set_xlabel('Ticker Symbol')
ax.set_ylabel('Number of Articles')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('./data/plot_ticker_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: plot_ticker_distribution.png')

## 6. PoC Query Demo — End-to-End Retrieval

Run a sample query through TF-IDF to show the pipeline works end-to-end.

In [ ]:
def tfidf_score(query: str, top_k: int = 10) -> pd.DataFrame:
    """Score all documents against a query using TF-IDF and return top-K results."""
    query_tokens = preprocess(query)
    scores = defaultdict(float)

    for term in query_tokens:
        if term not in inverted_index:
            continue
        term_idf = idf[term]
        for doc_id in inverted_index[term]:
            tf_val = doc_tf_index[doc_id][term]
            scores[doc_id] += tf_val * term_idf

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for doc_id, score in ranked:
        row = df.iloc[doc_id]
        results.append({
            'score':   round(score, 3),
            'ticker':  row[ticker_col],
            'date':    row[date_col],
            'title':   row[title_col],
            'snippet': str(row[article_col])[:150] + '...'
        })
    return pd.DataFrame(results)


# ── Run Example Queries (matching your P02 examples) ─────────────────────────
queries = [
    'semiconductor export restrictions',
    'supply chain disruption',
    'ransomware cyberattack data breach',
]

for q in queries:
    print(f'\n{"="*60}')
    print(f'QUERY: "{q}"')
    print('='*60)
    results = tfidf_score(q, top_k=5)
    print(results[['score', 'ticker', 'date', 'title']].to_string(index=False))

### TF-IDF Score Distribution for a Sample Query

In [ ]:
# Show the full score distribution for one query — validates that retrieval is discriminative
query = 'semiconductor export restrictions'
query_tokens = preprocess(query)
all_scores = defaultdict(float)

for term in query_tokens:
    if term not in inverted_index:
        continue
    for doc_id in inverted_index[term]:
        all_scores[doc_id] += doc_tf_index[doc_id][term] * idf[term]

score_values = list(all_scores.values())
nonzero_pct  = len(score_values) / N * 100

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(score_values, bins=60, color='darkorange', edgecolor='white', linewidth=0.3)
ax.set_title(f'TF-IDF Score Distribution for Query: "{query}"\n'
             f'({nonzero_pct:.1f}% of docs have a non-zero score — {len(score_values):,} docs)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('TF-IDF Score')
ax.set_ylabel('Number of Documents')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('./data/plot_query_score_dist.png', bbox_inches='tight')
plt.show()
print('Saved: plot_query_score_dist.png')

## 7. Summary Statistics for PDF Write-Up

In [ ]:
print('=' * 55)
print('SUMMARY — Copy these numbers into your P02 PDF')
print('=' * 55)
print(f'Sample size (rows):         {total_docs:,}')
print(f'Unique tickers:             {unique_tickers:,}')
print(f'Date range:                 {date_min.date()} to {date_max.date()}')
print(f'Missing/empty text:         {null_text:,} ({null_text/total_docs*100:.1f}%)')
print(f'Vocabulary size:            {len(corpus_tf):,} unique terms')
print(f'Hapax legomena:             {sum(1 for v in corpus_tf.values() if v==1):,} '
      f'({sum(1 for v in corpus_tf.values() if v==1)/len(corpus_tf)*100:.1f}%)')
print(f'Avg doc length (tokens):    {df["doc_length"].mean():.1f}')
print(f'Median doc length (tokens): {df["doc_length"].median():.1f}')
print(f'Max doc length (tokens):    {df["doc_length"].max():,}')
print(f'Inverted index size:        {len(inverted_index):,} postings lists')
print('=' * 55)
print('\nPlots saved to ./data/')
print('  plot_doc_length_dist.png')
print('  plot_tf_histogram.png')
print('  plot_idf_distribution.png')
print('  plot_top_terms.png')
print('  plot_ticker_distribution.png')
print('  plot_query_score_dist.png')